In [63]:
# ==============================================================================
# SCRIPT:         colab_vektor_engine_v1.9_definitive.ipynb
# OPERATION:      Operation: Vektor v1.9 (Pragmatic Drop Protocol)
# PURPOSE:        The definitive, correct, and pragmatic script. It intentionally
#                 and correctly drops rows with unparseable timestamps to
#                 guarantee a successful, clean, and near-complete output.
# ==============================================================================

# --- 1. SETUP ---
!pip install -q gspread folium
import pandas as pd
import numpy as np
import os
import gspread
import folium
from google.colab import auth, drive
from google.auth import default

print("Authenticating & Mounting...")
auth.authenticate_user(); creds, _ = default(); gc = gspread.authorize(creds)
drive.mount('/content/drive', force_remount=True)
print("✅ Setup complete.")

# --- 2. CONFIGURATION ---
ANCHORS_SPREADSHEET_NAME, ANCHORS_WORKSHEET_NAME = 'GTS-4', '250918'
TARGETS_SPREADSHEET_NAME, TARGETS_WORKSHEET_NAME = 'raw_Offers', 'raw_requests_VVSpolished'
ANCHOR_TS_COL, ANCHOR_LAT_COL, ANCHOR_LON_COL = 'realTimestamp', 'latitude', 'longitude'
TARGET_TS_COL, TARGET_ID_COL = 'timestamp', 'event_id'
SESSION_GAP_THRESHOLD_MINUTES = 30
OUTPUT_FOLDER_PATH = '/content/drive/MyDrive/_Pienza/Assets/Database/03_analysis'
OUTPUT_FILENAME = 'raw_Offers_with_Inferred_Vektor_DEFINITIVE2.csv'

# --- 3. BATTLE-TESTED & CORRECT UTILITY FUNCTIONS ---
def make_headers_unique(header_list):
    counts, unique_headers = {}, []
    for i, header in enumerate(header_list):
        header = str(header).strip() if str(header).strip() != "" else f"blank_{i}"
        if header not in counts:
            counts[header] = 1
            unique_headers.append(header)
        else:
            counts[header] += 1
            unique_headers.append(f"{header}_{counts[header]}")
    return unique_headers

def haversine_distance(lat1, lon1, lat2, lon2): R=6371000; phi1,phi2=np.radians(lat1),np.radians(lat2); dphi,dlambda=np.radians(lat2-lat1),np.radians(lon2-lon1); a=np.sin(dphi/2)**2+np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2; return R*2*np.arctan2(np.sqrt(a),np.sqrt(1-a))
def calculate_bearing(lat1, lon1, lat2, lon2): lat1,lon1,lat2,lon2=np.radians(lat1),np.radians(lon1),np.radians(lat2),np.radians(lon2); dLon=lon2-lon1; x=np.cos(lat2)*np.sin(dLon); y=np.cos(lat1)*np.sin(lat2)-np.sin(lat1)*np.cos(lat2)*np.cos(dLon); return (np.degrees(np.arctan2(x,y))+360)%360

try:
    # --- 4. STAGE 1: ROBUST Data Ingestion ---
    print("\n--- STAGE 1: ROBUST Data Ingestion ---")
    anchor_sheet=gc.open(ANCHORS_SPREADSHEET_NAME).worksheet(ANCHORS_WORKSHEET_NAME); df_anchors = pd.DataFrame(anchor_sheet.get_all_values()[1:], columns=make_headers_unique(anchor_sheet.get_all_values()[0])); df_anchors.replace('', np.nan, inplace=True); df_anchors['timestamp_utc']=pd.to_datetime(df_anchors[ANCHOR_TS_COL], errors='coerce'); df_anchors[ANCHOR_LAT_COL]=pd.to_numeric(df_anchors[ANCHOR_LAT_COL], errors='coerce'); df_anchors[ANCHOR_LON_COL]=pd.to_numeric(df_anchors[ANCHOR_LON_COL], errors='coerce'); df_anchors.dropna(subset=['timestamp_utc', ANCHOR_LAT_COL, ANCHOR_LON_COL], inplace=True); df_anchors['timestamp_utc']=df_anchors['timestamp_utc'].dt.tz_localize('UTC'); time_diff=df_anchors['timestamp_utc'].diff().dt.total_seconds()/60; session_breaks=time_diff>SESSION_GAP_THRESHOLD_MINUTES; df_anchors['session_id']=session_breaks.cumsum()

    target_sheet=gc.open(TARGETS_SPREADSHEET_NAME).worksheet(TARGETS_WORKSHEET_NAME); df_targets = pd.DataFrame(target_sheet.get_all_values()[1:], columns=make_headers_unique(target_sheet.get_all_values()[0])); df_targets.replace('', np.nan, inplace=True); df_targets['timestamp_utc']=pd.to_datetime(df_targets[TARGET_TS_COL], format='%Y:%m:%d %H:%M:%S', errors='coerce')
    initial_target_count = len(df_targets)

    # --- THIS IS THE RESTORED "PRAGMATIC DROP" LOGIC ---
    df_targets.dropna(subset=['timestamp_utc', TARGET_ID_COL], inplace=True)

    df_targets['timestamp_utc'] = df_targets['timestamp_utc'].dt.tz_localize('UTC'); df_targets.sort_values(by='timestamp_utc', inplace=True)
    print("  > Data ingestion and preparation successful.")

    # --- 5. STAGE 2: UNIFIED TIMELINE ---
    print("\n--- STAGE 2: Building Unified Timeline ---")
    df_anchors['source']='anchor'; df_targets['source']='target'
    df_unified=pd.concat([df_anchors, df_targets], ignore_index=True); df_unified.sort_values(by='timestamp_utc', inplace=True)
    df_unified['session_id'] = df_unified.groupby(df_unified['timestamp_utc'].dt.date)['session_id'].ffill().bfill()
    df_unified['anchor_lat']=np.where(df_unified['source']=='anchor',df_unified[ANCHOR_LAT_COL],np.nan); df_unified['anchor_lon']=np.where(df_unified['source']=='anchor',df_unified[ANCHOR_LON_COL],np.nan); df_unified['anchor_ts']=np.where(df_unified['source']=='anchor',df_unified['timestamp_utc'],pd.NaT)
    grouped=df_unified.groupby('session_id')
    for col in ['anchor_lat','anchor_lon','anchor_ts']: df_unified[f'{col}_before']=grouped[col].ffill(); df_unified[f'{col}_after']=grouped[col].bfill()
    df_final = df_unified[df_unified['source'] == 'target'].copy()

    # --- 6. STAGE 3 & 4: Vektor Calculation & Edge Cases ---
    print("\n--- STAGE 3 & 4: Calculating Vektors ---")
    time_total=(df_final['anchor_ts_after']-df_final['anchor_ts_before']).dt.total_seconds(); time_partial=(df_final['timestamp_utc']-df_final['anchor_ts_before']).dt.total_seconds(); ratio=(time_partial/time_total).clip(0,1).fillna(0)
    distance=haversine_distance(df_final['anchor_lat_before'],df_final['anchor_lon_before'],df_final['anchor_lat_after'],df_final['anchor_lon_after']); df_final['inferred_agent_speed_mps']=np.divide(distance,time_total); df_final['inferred_agent_bearing']=calculate_bearing(df_final['anchor_lat_before'],df_final['anchor_lon_before'],df_final['anchor_lat_after'],df_final['anchor_lon_after'])
    df_final.loc[time_total == 0, ['inferred_agent_speed_mps', 'inferred_agent_bearing']] = np.nan; df_final[['inferred_agent_speed_mps', 'inferred_agent_bearing']] = df_final.groupby('session_id')[['inferred_agent_speed_mps', 'inferred_agent_bearing']].transform(lambda x: x.bfill().ffill()); df_final.fillna({'inferred_agent_speed_mps':0}, inplace=True)
    df_final['inferred_agent_lat']=df_final['anchor_lat_before']+(df_final['anchor_lat_after']-df_final['anchor_lat_before'])*ratio; df_final['inferred_agent_lon']=df_final['anchor_lon_before']+(df_final['anchor_lon_after']-df_final['anchor_lon_before'])*ratio
    start,end,exact=df_final['anchor_lat_before'].isna(),df_final['anchor_lat_after'].isna(),(time_total==0)
    df_final.loc[start,['inferred_agent_lat','inferred_agent_lon']]=df_final.loc[start,['anchor_lat_after','anchor_lon_after']].values; df_final.loc[end,['inferred_agent_lat','inferred_agent_lon']]=df_final.loc[end,['anchor_lat_before','anchor_lon_before']].values; df_final.loc[start|end,['inferred_agent_speed_mps','inferred_agent_bearing']]=np.nan
    df_final['interpolation_quality']=np.select([start,end,exact],['EXTRAPOLATED_START','EXTRAPOLATED_END','EXACT_MATCH'],default='INTERPOLATED')

    # --- 7. STAGE 5: Save & Verification ---
    print("\n--- STAGE 5: Assembling Final Asset ---")
    final_row_count = len(df_final)
    cols_to_drop = [c for c in df_final.columns if c.startswith(('source', 'anchor_'))]
    df_final.drop(columns=cols_to_drop, inplace=True)

    full_output_path = os.path.join(OUTPUT_FOLDER_PATH, OUTPUT_FILENAME)
    df_final.to_csv(full_output_path, index=False)
    print(f"\n✅ Mission Success. Definitive asset with {final_row_count} rows saved to:\n   > '{full_output_path}'")
    if initial_target_count != final_row_count:
        print(f"  > INFO: {initial_target_count - final_row_count} rows were dropped due to missing or invalid timestamps, as per the pragmatic protocol.")

except Exception as e:
    import traceback; print(f"\n❌ An unrecoverable error occurred: {e}"); traceback.print_exc()

Authenticating & Mounting...
Mounted at /content/drive
✅ Setup complete.

--- STAGE 1: ROBUST Data Ingestion ---


/tmp/ipython-input-3669444205.py:54: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  target_sheet=gc.open(TARGETS_SPREADSHEET_NAME).worksheet(TARGETS_WORKSHEET_NAME); df_targets = pd.DataFrame(target_sheet.get_all_values()[1:], columns=make_headers_unique(target_sheet.get_all_values()[0])); df_targets.replace('', np.nan, inplace=True); df_targets['timestamp_utc']=pd.to_datetime(df_targets[TARGET_TS_COL], format='%Y:%m:%d %H:%M:%S', errors='coerce')


  > Data ingestion and preparation successful.

--- STAGE 2: Building Unified Timeline ---

--- STAGE 3 & 4: Calculating Vektors ---

--- STAGE 5: Assembling Final Asset ---

✅ Mission Success. Definitive asset with 4766 rows saved to:
   > '/content/drive/MyDrive/_Pienza/Assets/Database/03_analysis/raw_Offers_with_Inferred_Vektor_DEFINITIVE2.csv'
  > INFO: 10 rows were dropped due to missing or invalid timestamps, as per the pragmatic protocol.


In [29]:
# ==============================================================================
# SCRIPT:         Validation Stage 1: Bounding Box Sanity Check
# PURPOSE:        Programmatically verifies that all inferred locations fall
#                 within the expected geographic operational area.
# ==============================================================================
print("\n--- Validation Stage 1: Bounding Box Sanity Check ---")

try:
    # This is the final DataFrame that was successfully created in the previous cell.
    df_to_validate = df_final.copy()

    # Define the approximate geographic bounding box for the Mexico City operational area
    # These coordinates are generous but will easily catch gross errors.
    MIN_LAT, MAX_LAT = 19.0, 19.8  # Latitude for Mexico City area
    MIN_LON, MAX_LON = -99.5, -98.9 # Longitude for Mexico City area

    # Find outliers that are outside the box. We only check non-NaN values.
    outliers = df_to_validate.dropna(subset=['inferred_agent_lat', 'inferred_agent_lon'])[
        (df_to_validate['inferred_agent_lat'] < MIN_LAT) |
        (df_to_validate['inferred_agent_lat'] > MAX_LAT) |
        (df_to_validate['inferred_agent_lon'] < MIN_LON) |
        (df_to_validate['inferred_agent_lon'] > MAX_LON)
    ]

    if outliers.empty:
        print("✅ VERIFICATION PASSED: All inferred locations fall within the expected Mexico City bounding box.")
    else:
        print(f"❌ VERIFICATION FAILED: Found {len(outliers)} geographic outliers.")
        print("   These points require immediate investigation:")
        print(outliers[['event_id', 'timestamp', 'inferred_agent_lat', 'inferred_agent_lon', 'interpolation_quality']].to_string())

except NameError:
    print("❌ ERROR: The DataFrame 'df_final' was not found. Please ensure the main script cell was run successfully first.")
except Exception as e:
    print(f"❌ An error occurred during the sanity check: {e}")


--- Validation Stage 1: Bounding Box Sanity Check ---
✅ VERIFICATION PASSED: All inferred locations fall within the expected Mexico City bounding box.


/tmp/ipython-input-3297389241.py:18: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  outliers = df_to_validate.dropna(subset=['inferred_agent_lat', 'inferred_agent_lon'])[


In [30]:
# ==============================================================================
# SCRIPT:         Validation Stage 2: Visual Audit (v3.3 - Session-Aware)
# PURPOSE:        Generates a CORRECT visual audit map by creating a separate
#                 "ground-truth skeleton" for each individual session.
# ==============================================================================

print("\n--- Validation Stage 2: Visual Audit (Session-Aware) ---")

try:
    # These are the in-memory DataFrames from the initial ingestion.
    df_anchors_to_plot = df_anchors.copy()
    df_inferred_to_plot = df_final.copy()

    # Create a base map centered on Mexico City
    map_center = [19.4326, -99.1332]
    m = folium.Map(location=map_center, zoom_start=11, tiles="cartodbpositron")

    # --- The CRITICAL FIX: Group the skeleton by session ---
    print("Generating session-aware ground-truth skeleton (blue lines)...")
    # We group the anchors by the 'session_group' (date) we created during ingestion.
    session_paths = df_anchors_to_plot.groupby('session_group')

    # Iterate through each session and draw its own, separate line.
    for session_name, session_group in session_paths:
        points = session_group[['latitude', 'longitude']].values.tolist()
        folium.PolyLine(points, color='blue', weight=1.5, opacity=0.6).add_to(m)

    # --- Plot the Inferred Points (Red Dots) ---
    print("Overlaying inferred offer locations (red dots)...")
    # Plot all points now that we know they are within the bounding box.
    for _, row in df_inferred_to_plot.iterrows():
        if pd.notna(row['inferred_agent_lat']) and pd.notna(row['inferred_agent_lon']):
            # Color-code by quality
            if row['interpolation_quality'] == 'INTERPOLATED':
                dot_color = 'red'
            else: # EXTRAPOLATED
                dot_color = 'orange'

            folium.CircleMarker(
                location=[row['inferred_agent_lat'], row['inferred_agent_lon']],
                radius=2.5,
                color=dot_color,
                fill=True,
                fill_color=dot_color,
                fill_opacity=0.9,
                popup=f"Offer: {row['event_id']}<br>Quality: {row['interpolation_quality']}"
            ).add_to(m)

    # --- Save the map ---
    map_filename = 'trajectory_validation_map_DEFINITIVE.html'
    map_output_path = os.path.join(OUTPUT_FOLDER_PATH, map_filename)
    m.save(map_output_path)

    print(f"\n✅ VERIFICATION COMPLETE: The definitive, corrected map has been saved to your Google Drive.")
    print(f"   > '{map_output_path}'")
    print("   Please download and open this HTML file in your browser to visually inspect the results.")

except NameError:
    print("❌ ERROR: The DataFrames 'df_final' or 'df_anchors' were not found. Please ensure the main script cell ran successfully.")
except Exception as e:
    print(f"❌ An error occurred during map generation: {e}")


--- Validation Stage 2: Visual Audit (Session-Aware) ---
Generating session-aware ground-truth skeleton (blue lines)...
Overlaying inferred offer locations (red dots)...

✅ VERIFICATION COMPLETE: The definitive, corrected map has been saved to your Google Drive.
   > '/content/drive/MyDrive/_Pienza/Assets/Database/03_analysis/trajectory_validation_map_DEFINITIVE.html'
   Please download and open this HTML file in your browser to visually inspect the results.


In [32]:
# ==============================================================================
# SCRIPT:         Repair Protocol v1.8 (Definitive & Hardened)
# OPERATION:      Surgical Repair of Orphan Handling
# PURPOSE:        This definitive script surgically repairs the in-memory
#                 'df_merged' DataFrame, using robust methods to fix orphan
#                 rows and correct previous errors without making flawed
#                 assumptions about column structure.
# ==============================================================================
import pandas as pd
import numpy as np

print("\n--- INITIATING DEFINITIVE REPAIR PROTOCOL (v1.8) ---")

try:
    if 'df_merged' not in locals():
        raise NameError("The DataFrame 'df_merged' was not found. Please ensure the main script cell that created it has been run.")

    print("  > Located target DataFrame 'df_merged' in memory.")

    # --- STAGE 1: IDENTIFY FLAWED ROWS (UNCHANGED) ---
    start_orphans = df_merged['latitude_before'].isna()
    end_orphans = df_merged['latitude_after'].isna()
    orphan_indices = df_merged[start_orphans | end_orphans].index

    print(f"  > Identified {len(orphan_indices)} orphan records to repair.")

    # --- STAGE 2: SURGICAL REPAIR (NEW ROBUST LOGIC) ---
    print("  > Performing surgical repair on vector columns...")

    # For each orphan row, we explicitly set its inferred location to the single valid anchor it has.
    # This is more robust than fillna and recalculating a NaN ratio.

    # Repair start orphans
    df_merged.loc[start_orphans, 'inferred_agent_lat'] = df_merged.loc[start_orphans, 'latitude_after']
    df_merged.loc[start_orphans, 'inferred_agent_lon'] = df_merged.loc[start_orphans, 'longitude_after']

    # Repair end orphans
    df_merged.loc[end_orphans, 'inferred_agent_lat'] = df_merged.loc[end_orphans, 'latitude_before']
    df_merged.loc[end_orphans, 'inferred_agent_lon'] = df_merged.loc[end_orphans, 'longitude_before']

    # For ALL orphans, speed and bearing are definitively unknown.
    df_merged.loc[orphan_indices, 'inferred_agent_speed_mps'] = np.nan
    df_merged.loc[orphan_indices, 'inferred_agent_bearing'] = np.nan

    # The quality flag logic remains the same.
    df_merged['interpolation_quality'] = np.select([start_orphans, end_orphans], ['EXTRAPOLATED_START', 'EXTRAPOLATED_END'], default='INTERPOLATED')

    print("  > Repair successful. All blank vector rows have been populated.")

    # --- STAGE 3: Finalize and Save ---
    # We do NOT try to rebuild columns. We use the now-repaired df_merged as the source of truth.
    # The original 'df_targets' is no longer needed.

    # Define the columns we want to keep, based on what actually exists.
    final_cols_to_keep = [col for col in df_merged.columns if '_before' not in col and '_after' not in col]

    df_final_repaired = df_merged[final_cols_to_keep]

    repaired_output_path = os.path.join(OUTPUT_FOLDER_PATH, 'raw_Offers_with_Inferred_Vektor_DEFINITIVE.csv')
    df_final_repaired.to_csv(repaired_output_path, index=False)

    print(f"\n✅ REPAIR COMPLETE. Definitive asset has been saved to:\n   > '{repaired_output_path}'")

    print("\n--- Final Data Sample (Verification of Repaired Orphans) ---")
    print("Displaying the last rows of the repaired dataset:")
    print(df_final_repaired.tail().to_string())

except Exception as e:
    import traceback
    print(f"\n❌ An unrecoverable error occurred during the repair: {e}")
    traceback.print_exc()


--- INITIATING DEFINITIVE REPAIR PROTOCOL (v1.8) ---
  > Located target DataFrame 'df_merged' in memory.
  > Identified 56 orphan records to repair.
  > Performing surgical repair on vector columns...
  > Repair successful. All blank vector rows have been populated.

✅ REPAIR COMPLETE. Definitive asset has been saved to:
   > '/content/drive/MyDrive/_Pienza/Assets/Database/03_analysis/raw_Offers_with_Inferred_Vektor_DEFINITIVE.csv'

--- Final Data Sample (Verification of Repaired Orphans) ---
Displaying the last rows of the repaired dataset:
     Session_ID   Obs_ID       Date         Image Time Taken    Ride Type Exclusivo Es Exclusivo   Price Time to Pickup Distance to Pickup                                Pickup Location Est Time Est Distance                                                                                          Dropoff Location Rider Rating                     Special Note Decision1                   Reason Heuristic Flag         Decision2 Driver_State_at_Request

In [56]:
# ==============================================================================
# SCRIPT:         colab_final_join.ipynb (Definitive)
# OPERATION:      Operation Unification
# PURPOSE:        Performs the final, definitive left join to enrich the full
#                 'raw_Offers' dataset with the calculated Trajectory
#                 Intelligence Layer, preserving all original 4777 rows.
# ==============================================================================

# --- 1. SETUP ---
!pip install -q gspread
import pandas as pd
import numpy as np
import os
import gspread
from google.colab import auth, drive
from google.auth import default

print("Authenticating & Mounting...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
drive.mount('/content/drive', force_remount=True)
print("✅ Setup complete.")

# --- 2. CONFIGURATION ---
# The "Source of Truth" with 4777 rows
TRUTH_SPREADSHEET_NAME = 'raw_Offers'
TRUTH_WORKSHEET_NAME = 'raw_requests_VVSpolished'

# The "Intelligence Layer" with 4767 rows of vector data
VEKTOR_FILE_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis/raw_Offers_with_Inferred_Vektor_sessionaware.csv'

# Final Output Destination
OUTPUT_FOLDER_PATH = '/content/drive/My Drive/_Pienza/Assets/Database/03_analysis/'
OUTPUT_FILENAME = 'raw_Offers_FINAL_ENRICHED.csv'

# The Definitive Join Key
JOIN_KEY = 'event_id'

# The utility function to handle bad headers
def make_headers_unique(header_list):
    counts, unique_headers = {}, []; [counts.update({(h := str(header).strip() if str(header).strip() != "" else f"blank_{i}"): counts.get(h, 0) + 1}), unique_headers.append(f"{h}_{counts[h]}" if counts[h] > 1 else h)] for i, header in enumerate(header_list); return unique_headers

try:
    # --- 3. Load Both Data Assets ---
    print("\nLoading data sources...")

    # Load the "Source of Truth" from Google Sheets (all 4777 rows)
    worksheet = gc.open(TRUTH_SPREADSHEET_NAME).worksheet(TRUTH_WORKSHEET_NAME)
    all_values = worksheet.get_all_values()
    df_truth = pd.DataFrame(all_values[1:], columns=make_headers_unique(all_values[0]))
    df_truth.replace('', np.nan, inplace=True)
    print(f"  > Loaded {len(df_truth)} records from the original 'raw_Offers' sheet.")

    # Load the "Intelligence Layer" from the CSV
    df_vektor = pd.read_csv(VEKTOR_FILE_PATH)
    print(f"  > Loaded {len(df_vektor)} records from the Vektor file.")

    # Select only the key and the new columns from the Vektor data
    vektor_cols_to_join = [JOIN_KEY, 'inferred_agent_lat', 'inferred_agent_lon', 'inferred_agent_bearing', 'inferred_agent_speed_mps', 'interpolation_quality']
    df_vektor_subset = df_vektor[vektor_cols_to_join]

    # --- 4. Execute the Definitive LEFT JOIN ---
    print("\nPerforming definitive left join to enrich the full dataset...")
    df_final = pd.merge(
        left=df_truth,
        right=df_vektor_subset,
        on=JOIN_KEY,
        how='left'
    )
    final_row_count = len(df_final)
    print(f"  > Join complete. Final asset has {final_row_count} rows.")

    # --- 5. Diagnostic Protocol: Identify Dropped Rows ---
    print("\n--- Diagnostic Report: Identifying Unprocessed Rows ---")

    unprocessed_rows = df_final[df_final['inferred_agent_lat'].isnull()]

    if unprocessed_rows.empty:
        print("  > No unprocessed rows found. All 4777 records were successfully enriched.")
    else:
        print(f"  > Found {len(unprocessed_rows)} rows that were NOT processed by the Vektor script.")
        print("    This is likely due to missing or invalid 'timestamp' data in the original sheet.")
        print("    Details of unprocessed rows:")
        print(unprocessed_rows[['event_id', 'timestamp']].to_string())

    # --- 6. Save the Final, Definitive Asset ---
    full_output_path = os.path.join(OUTPUT_FOLDER_PATH, OUTPUT_FILENAME)
    df_final.to_csv(full_output_path, index=False)

    print(f"\n✅ Mission Success. The definitive, fully enriched, {final_row_count}-row asset has been saved to:")
    print(f"   > '{full_output_path}'")

except Exception as e:
    import traceback
    print(f"\n❌ An unrecoverable error occurred: {e}")
    traceback.print_exc()

SyntaxError: invalid syntax (ipython-input-331137726.py, line 42)